# Langfuse 프롬프트 관리 

## 환경 설정 및 준비

### (1) Env 환경변수

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### (2) 기본 라이브러리

In [2]:
import os
from glob import glob
from pprint import pprint
import json
import warnings
warnings.filterwarnings("ignore")

### (3) Langfuse 콜백 핸들러 설정

In [3]:
from langfuse.langchain import CallbackHandler 

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

### (4) Langfuse 클라이언트 설정

In [4]:
from langfuse import get_client

# Langfuse 클라이언트 초기화
langfuse = get_client()

# 연결 테스트
assert langfuse.auth_check()

---

## 프롬프트 관리 개요

Langfuse는 **프롬프트 CMS(Content Management System)** 기능을 제공

- **버전 관리**: 프롬프트의 모든 변경사항을 추적하고 롤백 가능
- **협업**: 팀원들과 함께 프롬프트를 편집하고 관리
- **배포 관리**: 라벨을 통해 코드 변경 없이 환경별 배포
- **성능 모니터링**: 프롬프트 버전별 성능 메트릭 비교
- **실시간 테스트**: 플레이그라운드에서 즉시 테스트 가능

---

## 1. 프롬프트 생성

### 1.1 텍스트 프롬프트 생성

In [5]:
# 텍스트 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic",  # 프롬프트 이름
    type="text",          
    prompt="{{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?",
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

### 1.2 챗 프롬프트 생성

In [ ]:
# 챗 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic-chat",  # 프롬프트 이름
    type="chat",          
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}를 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "chat"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

### 1.3 메시지 플레이스홀더가 있는 챗 프롬프트

In [ ]:
# 메시지 플레이스홀더를 포함한 챗 프롬프트
langfuse.create_prompt(
    name="movie-critic-with-history",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "type": "placeholder",
            "name": "chat_history"  # 대화 히스토리 삽입 지점
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대해 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],
    tags=["movie", "qa", "chat", "history"]
)

### **[실습 1]**
텍스트 기반 프롬프트와 chat 기반 프롬프트를 각각 구현하고, Langfuse UI에서 확인하세요.

In [6]:
# 텍스트 프롬프트 생성
# 여기에 코드를 작성하세요
langfuse.create_prompt(
    name="movie-critic",  # 프롬프트 이름
    type="text",          
    prompt="{{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?",
    # labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

In [15]:
# 챗 프롬프트 생성
# 여기에 코드를 작성하세요
langfuse.create_prompt(
    name="movie-critic-with-history",  # 프롬프트 이름
    type="chat",          
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}를 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "chat"],    # 태그
    config={
        "model": "gpt-4.1-mini",
        "temperature": 0.7,
        "max_tokens": 500
    }
)

---

## 2. 프롬프트 활용

### 2.1 기본 프롬프트 가져오기

In [8]:
# 프로덕션 버전 가져오기
prompt = langfuse.get_prompt("movie-critic")

# 프롬프트 정보 출력
print(f"모델: {prompt.config['model']}")
print(f"온도: {prompt.config['temperature']}")
print(f"라벨: {prompt.labels}")
print(f"태그: {prompt.tags}")
print(f"프롬프트: {prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['production']
태그: ['movie', 'qa', 'text']
프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
----------------------------------------------------------------------------------------------------
{criticLevel} 영화 평론가로서, {movie}를 어떻게 생각하시나요?


### 2.2 compile 메서드 사용

- compile 메서드로 변수 삽입

In [9]:
# compile 메서드로 변수 삽입
compiled_prompt = prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_prompt)

전문가 영화 평론가로서, 인셉션를 어떻게 생각하시나요?


### 2.3 챗 프롬프트 가져오기 및 컴파일

In [11]:
# 챗 프롬프트 가져오기
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 챗 프롬프트 정보 출력
print(f"모델: {chat_prompt.config['model']}")
print(f"온도: {chat_prompt.config['temperature']}")
print(f"라벨: {chat_prompt.labels}")
print(f"태그: {chat_prompt.tags}")
print(f"프롬프트: {chat_prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(chat_prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['latest', 'production']
태그: ['movie', 'qa', 'chat']
프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
[('system', '당신은 {criticLevel} 영화 평론가입니다.'), ('user', '영화 {movie}를 어떻게 생각하시나요?')]


In [12]:
# 챗 프롬프트 컴파일
compiled_chat_prompt = chat_prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_chat_prompt)

[{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}, {'role': 'user', 'content': '영화 인셉션를 어떻게 생각하시나요?'}]


### 2.4 메시지 플레이스홀더 활용

In [16]:
# 플레이스홀더가 있는 챗 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# 대화 히스토리 정의
chat_history = [
    {"role": "user", "content": "안녕하세요!"},
    {"role": "assistant", "content": "안녕하세요! 영화에 대해 이야기해볼까요?"}
]

# 변수와 플레이스홀더를 모두 컴파일
compiled_with_history = prompt_with_history.compile(
            criticLevel="전문가",
            movie="인셉션", 
            chat_history=chat_history
        )

for message in compiled_with_history:
    print(message)
    print("-" * 20)

{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}
--------------------
{'role': 'user', 'content': '영화 인셉션를 어떻게 생각하시나요?'}
--------------------


### **[실습 2]**
"movie-critic-chat" 프롬프트를 Langfuse에서 가져와서 내용을 출력하고, compile 메서드를 사용해 변수에 적절한 값을 추가해보세요.

In [ ]:
# chat 프롬프트 가져오기 및 컴파일
# 여기에 코드를 작성하세요

In [ ]:
# chat 프롬프트 가져오기 및 컴파일
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 대화 히스토리 정의
chat_history = [
    {"role": "user", "content": "안녕하세요!"},
    {"role": "assistant", "content": "안녕하세요! 영화에 대해 이야기해볼까요?"}
]

# 변수와 플레이스홀더를 모두 컴파일
compiled_chat_prompt = chat_prompt.compile(
    criticLevel="전문가", 
    movie="인셉션"
)

for message in compiled_chat_prompt:
    print(message)
    print("-" * 20)

---

## 3. 프롬프트 버전 관리

### 3.1 새로운 버전 생성

In [ ]:
# 새로운 버전 생성 (같은 이름 사용)
langfuse.create_prompt(
    name="movie-critic",  # 같은 이름 사용
    type="text",          
    prompt="당신은 {{criticLevel}} 영화 평론가입니다.\n\n영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.",
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text", "detailed"],    # 태그 업데이트
    config={
        "model": "gpt-4.1",  # 모델 업그레이드
        "temperature": 0.7,
        "max_tokens": 1000  # 토큰 수 증가
    }
)

### 3.2 특정 버전 가져오기

In [ ]:
# 특정 버전 가져오기
prompt_v1 = langfuse.get_prompt("movie-critic", version=1)
prompt_v2 = langfuse.get_prompt("movie-critic", version=2)

# 버전별 비교
print(f"V1 프롬프트: {prompt_v1.prompt}")
print(f"V2 프롬프트: {prompt_v2.prompt}")
print(f"V1 모델: {prompt_v1.config['model']}")
print(f"V2 모델: {prompt_v2.config['model']}")

### 3.3 라벨 관리

In [ ]:
# 특정 라벨로 프롬프트 생성 (같은 이름을 사용하면 새로운 버전으로 생성됨)
langfuse.create_prompt(
    name="movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)

# 라벨별 프롬프트 가져오기
prompt_production = langfuse.get_prompt("movie-critic-chat", label="production")
prompt_staging = langfuse.get_prompt("movie-critic-chat", label="staging")
prompt_latest = langfuse.get_prompt("movie-critic-chat", label="latest")

In [ ]:
# 라벨별 프롬프트 출력
print(f"Production 프롬프트: {prompt_production.prompt}")
print("-" * 100)
print(f"Staging 프롬프트: {prompt_staging.prompt}")
print("-" * 100)
print(f"Latest 프롬프트: {prompt_latest.prompt}")

### 3.4 라벨 업데이트

In [ ]:
# 기존 프롬프트 버전의 라벨 업데이트
langfuse.update_prompt(
    name="movie-critic-chat",
    version=2,
    new_labels=["production", "v2-stable"]
)

### **[실습 3]**
"movie-critic-chat" 프롬프트를 수정하고, labels 속성은 "staging"으로 지정한 후, staging 버전을 가져와서 내용을 출력하세요.

In [ ]:
# staging 라벨 생성
# 여기에 코드를 작성하세요

# staging 라벨 가져오기
# 여기에 코드를 작성하세요

In [ ]:
# staging 라벨 생성
# 여기에 코드를 작성하세요
langfuse.create_prompt(
    name="movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 일반적이고 상세하며 전문적인 분석을 제공해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)

# staging 라벨 가져오기
prompt_staging = langfuse.get_prompt(
    name="movie-critic-chat",
    label="staging"
)

# 라벨별 프롬프트 출력
print(f"Staging 프롬프트: {prompt_staging.prompt}")

---

## 4. LangChain과의 통합

### 4.1 텍스트 프롬프트와 LangChain 통합

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Langfuse 프롬프트를 LangChain과 통합
prompt = langfuse.get_prompt("movie-critic", label="production")

langchain_prompt = PromptTemplate.from_template(
    prompt.get_langchain_prompt(),
    metadata={"langfuse_prompt": prompt},  # Langfuse 자동 링크를 위한 메타데이터
)

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.7),
    max_completion_tokens=prompt.config.get("max_tokens", 500)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
)

print(response.content)

### 4.2 챗 프롬프트와 LangChain 통합

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 챗 프롬프트 통합
chat_prompt = langfuse.get_prompt("movie-critic-chat", label="production", type="chat")

langchain_chat_prompt = ChatPromptTemplate.from_messages(
    chat_prompt.get_langchain_prompt()
)
langchain_chat_prompt.metadata = {"langfuse_prompt": chat_prompt}

# 체인 실행
chain = langchain_chat_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}
)

print(response.content)

### 4.3 플레이스홀더가 있는 프롬프트와 LangChain 통합

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

# 플레이스홀더가 있는 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# LangChain 호환 프롬프트로 변환 (미해결 플레이스홀더는 MessagesPlaceholder로 변환)
langchain_prompt_with_placeholder = ChatPromptTemplate.from_messages(
    prompt_with_history.get_langchain_prompt()
)
langchain_prompt_with_placeholder.metadata = {"langfuse_prompt": prompt_with_history}

# chain 생성
chain = langchain_prompt_with_placeholder | model

# 실행 시 플레이스홀더 값 제공
chat_history = [
    {"role": "user", "content": "안녕하세요! 영화 예산에 대해서 이야기해볼까요?"},
    {"role": "assistant", "content": "안녕하세요! 영화 예산에 대해 어떻게 도와드릴까요?"}
]

response = chain.invoke({
    "criticLevel": "전문가", 
    "movie": "인셉션",
    "chat_history": chat_history
}, config={"callbacks": [langfuse_handler]})  # Langfuse 트레이싱을 위한 콜백

print(response.content)

### **[실습 4]**
앞에서 정의한 텍스트 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.


In [17]:
# 특정 라벨 가져오기
prompt_staging = langfuse.get_prompt("movie-critic", label="latest")  # production, latest

# 프롬프트 출력
print(f"모델: {prompt_staging.config['model']}")
print(f"온도: {prompt_staging.config['temperature']}")
print(f"라벨: {prompt_staging.labels}")
print(f"프롬프트: {prompt_staging.prompt}")

모델: gpt-4.1-mini
온도: 0.7
라벨: ['latest']
프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?


In [18]:
from langchain_core.prompts import ChatPromptTemplate

# Langchain과 통합 - 'chat' 프롬프트
langchain_prompt = ChatPromptTemplate.from_template(
    prompt_staging.get_langchain_prompt(type="chat"),
)
langchain_prompt.metadata = {"langfuse_prompt": prompt_staging}

print(langchain_prompt.format(criticLevel="비평가", movie="인셉션"))

Human: 비평가 영화 평론가로서, 인셉션를 어떻게 생각하시나요?


In [19]:
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델 초기화 (프롬프트 설정에서 가져온 값 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.7)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # 콜백 핸들러 추가
)

# 응답 출력
print(response.content)

크리스토퍼 놀란 감독의 **<인셉션> (Inception, 2010)**은 현대 영화사에서 매우 중요한 위치를 차지하는 작품이라고 생각합니다. 이 영화는 단순한 대중적인 오락을 넘어, 복잡한 서사 구조와 철학적 주제를 결합한 점에서 뛰어난 예술적 성취를 보여줍니다.

첫째, **서사적 혁신과 구조의 복합성**이 돋보입니다. 꿈 속의 꿈이라는 다중 레이어 구조를 통해 시간과 현실의 경계를 흐릿하게 만들며, 관객에게 끊임없는 긴장감과 몰입을 유도합니다. 이러한 내러티브는 전통적인 플롯 진행과는 달리, 퍼즐을 맞추듯 관객이 적극적으로 이야기의 의미를 해석하게 만듭니다.

둘째, **시각적·기술적 완성도** 역시 매우 인상적입니다. 실제로 물리 법칙이 뒤틀리는 듯한 장면들, 중력을 무시한 액션 시퀀스 등은 CG와 실사 촬영을 절묘하게 조화시켜, 시각적 쾌감을 극대화합니다. 한편, 한스 짐머의 음악은 긴장감과 감정선을 효과적으로 이끌어내며 영화의 몰입도를 높이는 데 크게 기여합니다.

셋째, **주제적 깊이**도 빼놓을 수 없습니다. 현실과 꿈, 기억과 죄책감, 인간의 무의식과 정체성 등 심리적이고 철학적인 질문을 던지며, 단순한 SF 액션을 넘어 인간 내면의 복잡성을 탐구합니다. 특히 도미닉 코브(레오나르도 디카프리오 분)의 개인적 상처와 구원에 대한 이야기는 감정적 울림을 더합니다.

결론적으로, <인셉션>은 뛰어난 상업성과 예술성을 동시에 갖춘 작품으로, 현대 영화에서 ‘블록버스터’와 ‘예술 영화’의 경계를 허무는 중요한 사례입니다. 영화 산업뿐 아니라 대중문화 전반에 미친 영향력도 상당하며, 앞으로도 오랫동안 분석되고 회자될 명작이라 평가합니다.


### **[실습 5]**
앞에서 정의한 chat 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.

In [ ]:
# 특정 라벨 가져오기
prompt_staging = langfuse.get_prompt("movie-critic-chat", label="latest", type="chat")  # production, latest

# 프롬프트 출력
print(f"모델: {prompt_staging.config['model']}")
print(f"온도: {prompt_staging.config['temperature']}")
print(f"라벨: {prompt_staging.labels}")
print(f"프롬프트: {prompt_staging.prompt}")

from langchain_core.prompts import ChatPromptTemplate

# Langchain과 통합 - 'chat' 프롬프트
langchain_prompt = ChatPromptTemplate.from_messages(
    prompt_staging.get_langchain_prompt(),
)
langchain_prompt.metadata = {"langfuse_prompt": prompt_staging}

print(langchain_prompt.format(criticLevel="비평가", movie="인셉션"))

from langchain_openai import ChatOpenAI

# ChatOpenAI 모델 초기화 (프롬프트 설정에서 가져온 값 사용)
model = ChatOpenAI(
    model=prompt_staging.config.get("model", "gpt-4.1-mini"),
    temperature=prompt_staging.config.get("temperature", 0.7)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # 콜백 핸들러 추가
)

# 응답 출력
print(response.content)

모델: gpt-4.1-mini
온도: 0.7
라벨: ['latest', 'production']
프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
System: 당신은 비평가 영화 평론가입니다.
Human: 영화 인셉션를 어떻게 생각하시나요?
영화 **인셉션(Inception, 2010)**은 크리스토퍼 놀란 감독의 대표작 중 하나로, 현대 영화사에서 매우 중요한 위치를 차지하는 작품이라고 생각합니다. 

첫째, **스토리텔링의 혁신성**이 돋보입니다. 꿈과 현실의 경계를 넘나드는 복잡한 플롯은 관객들에게 깊은 몰입감을 선사하며, 여러 층위의 꿈 속에서 벌어지는 사건들이 정교하게 얽혀 있어 반복해서 봐도 새로운 해석이 가능합니다.

둘째, **시각적 효과와 연출**이 뛰어납니다. 현실과 꿈의 세계를 구분 짓는 독창적인 시각 효과와 액션 시퀀스는 당시 기술적 한계를 뛰어넘는 수준이었고, 지금까지도 많은 영화들이 본받는 기준이 되고 있습니다.

셋째, **음악과 사운드 디자인** 역시 영화의 긴장감과 몰입도를 높이는 데 큰 역할을 했습니다. 한스 짐머의 음악은 특히 인상적이며, 영화의 테마와 감정을 강렬하게 전달합니다.

마지막으로, **주제적 깊이**도 빼놓을 수 없습니다. 꿈과 현실, 기억과 죄책감, 자아 탐색 등 철학적이고 심리적인 요소들이 복합적으로 작용하여 단순한 액션 스릴러를 넘어선 작품으로 평가받습니다.

종합적으로 보면, 인셉션은 기술적 완성도와 예술적 깊이를 모두 갖춘 영화로, 현대 영화 팬들과 평론가들 사이에서 꾸준히 회자되는 명작이라고 할 수 있습니다.


: 